In [ ]:
print("all ok")

The Goal
You've now seen 6 different providers. Notice how similar they all look? Let's take that observation and build a single function that handles all of them.

Why is this useful?

Switch providers by changing one string: call_llm("gemini", ...) → call_llm("openai", ...)

In [ ]:
import os

for key in ("OPENAI_API_KEY", "OPENROUTER_API_KEY","HUGGINGFACEHUB_API_TOKEN","GOOGLE_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    print(f"{key} configured: {bool(value)}")
    print(value)

In [ ]:
# 👇 UNIVERSAL LLM CALLER
#    One function that works with any of the 6 providers.
#    Fill in the ___ parts using what you learned in Section 1.
from openai import OpenAI
def call_llm(provider: str, prompt: str, api_key: str = "", model: str = "") -> str:
    """
    Call any LLM provider with the same interface.

    Args:
        provider : One of: "ollama" | "lmstudio" | "openai" | "anthropic" | "gemini" | "openrouter"
        prompt   : The question or instruction to send to the model
        api_key  : Your API key (leave empty for local providers Ollama and LM Studio)
        model    : Model name — if empty, a sensible default is used for each provider

    Returns:
        The model's response as a plain Python string
    """
    
    # ------------------------------------------------------------------ #
    #  OPENAI — cloud API at api.openai.com                              #
    # ------------------------------------------------------------------ #
    
    
    if provider == "openai":
        OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
        if not OPEN_API_KEY:
            raise ValueError("OPENAI_API_KEY environment variable not set")
        print("Calling OpenAI...")
        client = OpenAI(api_key=OPEN_API_KEY)   # TODO: pass the api_key parameter to the client
        model = model or "gpt-4o-mini"  # cheapest GPT-4 class model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500
        )
        return resp.choices[0].message.content
    
    # ------------------------------------------------------------------ #
    #  OPENROUTER — cloud gateway to 200+ models                          #
    # ------------------------------------------------------------------ #
    
    if provider == "openrouter":
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if not OPENROUTER_API_KEY:
            raise ValueError("OPENROUTER_API_KEY environment variable not set")
        print("Calling OpenRouter...")
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",    # TODO: OpenRouter's base URL (e.g. "https://openrouter.ai/api/v1")
            api_key=OPENROUTER_API_KEY,
        )
        model = model or "meta-llama/llama-3.3-70b-instruct:free"   # free 70B model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return resp.choices[0].message.content   # TODO: return the response text (same pattern as OpenAI) 
    
    # ------------------------------------------------------------------ #
    #  GEMINI — Google's cloud API, its own SDK                           #
    # ------------------------------------------------------------------ #
    
    elif provider == "gemini":
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if not GOOGLE_API_KEY:
            raise ValueError("GOOGLE_API_KEY environment variable not set")
        print("Calling Google Gemini...")
        from google import genai
        client_gemini = genai.Client(api_key=GOOGLE_API_KEY)
        model_name = model or "gemini-2.0-flash"
        response = client_gemini.models.generate_content(
            model=model_name,
            contents=[{"type": "input_text", "input_text": {"text": prompt}}],
            )
        # return the text field (fallback to str(response) if attribute missing)
        return getattr(response, "text", None) or str(response)
    
    else:
        raise ValueError(
            f"Unknown provider: '{provider}'. "
            f"Choose from: ollama, lmstudio, openai, anthropic, gemini, openrouter"
        )


# 👇 Quick smoke test — run this to verify Ollama works
#    (swap "ollama" for "gemini" or "openrouter" if you're on Colab)
#result = call_llm("openrouter", "What is 2 + 2? Answer with one word only.")
#print(f"Test passed! openai says: {result}")
        
        
        

SECTION 3 — EXPERIMENT: Compare Responses Across Providers
What we're testing
Now that call_llm() works with any provider, let's use it to run the same question across multiple providers and compare:

The quality of answers — are they accurate? Complete?
The speed — how long does each provider take?
The style — do different models have different personalities?

In [ ]:
import time

# 👇 The SAME question will be sent to every provider you configure below
QUESTION = "What is the most important thing to understand about large language models?"
MY_PROVIDERS = {
    #"openai": "gpt-4o-mini",
    #"openrouter": "meta-llama/llama-3.3-70b-instruct:free",
    "gemini": "gemini-2.0-flash"
}

print(f"Question: {QUESTION}")
print("=" * 60)

for provider, model in MY_PROVIDERS.items():
    print(f"\n=== Testing {provider} with model {model} ===")
    start_time = time.time()
    try:
        answer = call_llm(provider, QUESTION, model=model)
        elapsed = time.time() - start_time
        print(f"Answer from {provider} (took {elapsed:.2f} seconds):\n{answer}")
    except Exception as e:
        print(f"Error calling {provider}: {e}")

Using OPENAPI SDK

#### Claude via LangChain
Use Anthropic/Claude with `ANTHROPIC_API_KEY` and LangChain.

In [ ]:
import os
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0
)

# Text to Image

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")

In [ ]:
kwargs = {"type": "image_generation", "quality": "low"}

In [ ]:
llm_with_tools =llm.bind_tools([kwargs])

In [ ]:
ai_message = llm_with_tools.invoke("Draw a picture of a cute fuzzy cat with an umbrella")  

In [ ]:
ai_message

In [ ]:
import base64
from IPython.display import Image

In [ ]:
image = next(
    item for item in ai_message.content_blocks if item["type"] == "image"
)

In [ ]:
Image(base64.b64decode(image["base64"]), width=200)

# Image to Text

#### Model from  OPENAI

#### OCR(optical character recognition)

##### to fetch a data from the images

#### it is cv based technique 

#### ealier we were performing it using the CNN based model

#### Now we can perform OCR task using the Transformer based model(ViT)

#### These models plays a very imp role in MMRAG Pipeline

## using this base64 encode we can perform bindary encoding of our data(image, audio, video)

### system is percoessing the data in binary form

#### image(collection of pixel) is a data

#### image required encoding and we are converting image image into the binary format(base64)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
base64_image = encode_image("D:\\Full_stack_GenAI\\Repo\\Full-Stack-GenAI\\Class-16-16-May-2026\\dogs.png")

In [ ]:
base64_image

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
message = HumanMessage(
    content=[
        {"type": "text", "text": "What is in this image?"},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{base64_image}"
            },
        },
    ]
)

In [ ]:
message = HumanMessage(
    content=[
        {"type": "text", "text": "What is in this image?"},
        {
            "type": "image_url",
            "image_url": {
                "url": "https://www.coolutils.com/blog/wp-content/uploads/2025/06/jpg.png"
            },
        },
    ]
)

In [ ]:
response = llm.invoke([message])


In [ ]:
print(response.content)

The image features a playful illustration of six cartoon-style dogs. Each dog has a distinct expression, often with happy faces and their tongues out. The dogs vary in breed and fur textures, adding a fun and whimsical touch to the overall composition. The background is a solid color that contrasts nicely with the colorful dogs.

The image appears to be a graphic that features a question, "What Is a JPG File?" along with an icon representing a JPG file. The icon likely includes imagery such as a document with a picture symbol and the letters "JPG" prominently displayed, all set against a colorful background.

scratch --> limitation

So far we have seen OpenAI which is paid one
Anthropic is also paid
you can purchase their credits 
config the api key
and run it via langchain
Now lets access some free models
text to text models
text to image models
image to text models 
are also possible
1. GROQ --? free access
2. Openrouter --? free access
3. Google(gemini api) --> here we have limited access of limited models
4. Huggingface Inferencing API

HuggingfaceHUB
OllamaHUB

## Groq
## Text to text: free

In [ ]:
import os
GROQ_API_KEY = os.getenv ("GROQ_API_KEY")  # This will return None if the environment variable is not set
if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3-32b")

In [ ]:
## this message which we are passsing it is only called prompt or input 
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Hindi. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]

In [ ]:
ai_msg = llm.invoke(messages)

In [ ]:
print(ai_msg.content)

#### Groq

### Image to text: free



In [ ]:
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct"
)

In [ ]:
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Describe this image in detail."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "https://www.coolutils.com/blog/wp-content/uploads/2025/06/jpg.png"
            }
        }
    ]
)

In [ ]:
response = llm.invoke([message])

In [ ]:
response

In [ ]:
print(response.content)

  ### Openrouter
  #### text to text: free

In [ ]:
import os
OPENROUTER_API_KEY = os.getenv ("OPENROUTER_API_KEY")  # This will return None if the environment variable is not set
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("OPENROUTER_API_KEY configured:", bool(OPENROUTER_API_KEY))

In [ ]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",   # OpenRouter's cloud endpoint
    api_key=OPENROUTER_API_KEY,
)

# 👇 ":free" at the end of the model name means it costs nothing
#    You get a 70-billion parameter model for free — remarkable!
response = openrouter_client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct:free",   # 70B parameter model, completely free
    messages=[
        {
            "role": "user",
            "content": "What is quantization in LLMs? Explain in 2-3 sentences."
        },
    ],
    max_tokens=300,
)

# 👇 Same response shape as OpenAI — identical code, different provider
print(response.choices[0].message.content)
print(f"\nModel: {response.model}")

#### via openrouter you can access the claude model

#### but guys here we have limitation (in the free versio)

#### the limitation is we can only access 2659 tokens in input/output

#### free version having a limit till max_tokens=2659

In [ ]:
from langchain_openrouter import ChatOpenRouter

In [ ]:
llm = ChatOpenRouter(model="anthropic/claude-sonnet-4.5",max_tokens=2048)

In [ ]:
messages = [
    (
        "system",
        "You are a helpful assistant.",
    ),
    ("human", "can you talk about the climate change?."),
]

In [ ]:
ai_msg = llm.invoke(messages)

In [ ]:
print(ai_msg.content)

### checkout all the models provided by openrouter over here : https://openrouter.ai/models?input_modalities=text,image

##### as a homework you can practice or experiment with the different different model from the openrouter

#### huggingface

### text to text: free

In [ ]:
import os
HUGGINGFACEHUB_API_TOKEN = os.getenv ("HUGGINGFACEHUB_API_TOKEN")
print("HUGGINGFACEHUB_API_TOKEN configured:", bool(HUGGINGFACEHUB_API_TOKEN))# This will return None if the environment variable is not set
if HUGGINGFACEHUB_API_TOKEN:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="text-generation",
    max_new_tokens=512
)

chat_model = ChatHuggingFace(llm=llm)

messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(content="What happens when an unstoppable force meets an immovable object?")
]

response = chat_model.invoke(messages)

print(response.content)




Prompt(Input message) ---->LLM----->output(output message or AI-Message)

user_message

or

system_message
user_message

we have two way of defining ths system and user message

1. first simple tuple format(older way of passing message but still relevant)

messages = [
    ("system", "You are a helpful assistant that translates English to Hindi. Translate the user sentence.",),
    ("human", "I love programming.")
     ]

2. second using SystemMessage class and HumanMessage class(latest way of defining message)
latest LangChain variant
messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage( content="What happens when an unstoppable force meets an immovable object?"),
]

SystemMessage or system -->we are defining the permanent behviour of llm in current active session
HumanMessage or human --> whatever human wanted to provide input it is that message

SystemMessage just to set the role or the behaviour of the LLM for the active session.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(content="What happens when an unstoppable force meets an immovable object?")
]

In [ ]:
ai_msg = chat_model.invoke(messages)

In [ ]:
print(ai_msg.content)

### as a homework you can experiment with the different models from the huggingface Text to Image, Image to text, Audio to text, text to Audio

### gemini(google)

### text to text : free tier available

Model: gemini-2.5-flash (free)

Model: gemini-2.5-pro(free)

Model: gemini-2.5-flash-lite(free)

Model: gemini-2.5-flash-native-audio-preview(free)

Model: gemini-2.5-flash-preview-tts(free)

Model: gemini-embedding-2(free)

Model: gemini-embedding-001(free)



In [ ]:
import os
GOOGLE_API_KEY = os.getenv ("GOOGLE_API_KEY")  # This will return None if the environment variable is not set
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print(GOOGLE_API_KEY)

## Text to Text model

In [ ]:
from google import genai

client_gemini = genai.Client(api_key=GOOGLE_API_KEY)

response = client_gemini.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Explain the conept of attention in 3 bullet points.",
    config = genai.types.GenerateContentConfig(
        max_output_tokens=300,
        temperature=0.7,
    )
)
    
print(response.text)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite"
)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
messages = [
        SystemMessage(content="Translate the following English text to Tamil"),
        HumanMessage(content="Pure tamil names for baby boy")
    ]

In [ ]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]

In [ ]:
ai_msg = llm.invoke(messages)

In [ ]:
print(ai_msg.content)

### google gemini

## Image to text

In [ ]:
import base64
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# 1. Image ko base64 me convert karo
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

base64_image = encode_image("D:\\Full_stack_GenAI\\Repo\\Full-Stack-GenAI\\Class-16-16-May-2026\\dogs.png")

In [ ]:
# 2. Gemini model initialize karo
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
# 3. Text + image message banao
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Describe this image in detail."
        },
        {
            "type": "image_url",
            "image_url": f"data:image/png;base64,{base64_image}"
        }
    ]
)

In [ ]:
# 4. Model invoke karo
response = llm.invoke([message])

In [ ]:
print(response.content)

# Audio → Text

In [ ]:
import base64
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [ ]:
def encode_audio(audio_path):
    with open(audio_path, "rb") as audio_file:
        return base64.b64encode(audio_file.read()).decode("utf-8")

In [ ]:
base64_audio = encode_audio("D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-16-16-May-2026\\audio.mp3")

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Transcribe this audio into English text."
        },
        {
            "type": "media",
            "mime_type": "audio/mp3",
            "data": base64_audio
        }
    ]
)

In [ ]:
response = llm.invoke([message])

In [ ]:
print(response.content)

Hello students. Today we are learning about multimodal AI, one of the most exciting areas in artificial intelligence, where models can understand and generate multiple types of data such as text, images, audio, and video, enabling powerful applications like image captioning, speech recognition, text-to-image generation, AI voice assistants, visual question answering, and intelligent agentic systems that can interact with the world in a much more human-like way.

### Text → Audio

here we are going to convert text to audio via gemini model
you can convert using the different other model also(like openai is also providing you some models)
some model you will get from huggingface/openrouter...

# audio to audio
# Image to image
# video to video
# text to audio,video

but for these kind of task you will not always get langchain wrapper

### if dont have langchain wrapper so we can directly use those model via API SDK

so we can directly use openai sdk/ google gemini sdk/ openrouter sdk

In [ ]:
from google import genai
from google.genai import types
import wave

In [ ]:
client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts", #freely accessible model for text to speech conversion
    contents="Hello students, today we are learning multimodal AI.and you are in sunny savita class wheere he is teaching you multimodal ai.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name="Kore"
                )
            )
        )
    )
)

In [ ]:
audio_data = response.candidates[0].content.parts[0].inline_data.data

with wave.open("output.wav", "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(audio_data)

print("Audio saved as output.wav")

### 

## Ollama

ollama --version
ollama help

ollama serve

ollama pull llama3.1

ollama run llama3.1

ollama list

ollama ps

ollama stop llama3.1

ollama show llama3.1

ollama show llama3.1 --modelfile

ollama rm llama3.1

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# 1. Create local Ollama LLM
llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

In [ ]:
# 2. Create messages
messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="Explain RAG in simple words.")
]

In [ ]:
# 3. Invoke model
response = llm.invoke(messages)

In [ ]:
# 4. Print output
print(response.content)

In [ ]:
!ollama pull bakllava

In [ ]:
import base64
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

def encode_image(image_path: str) -> str:
    image_bytes = Path(image_path).read_bytes()
    return base64.b64encode(image_bytes).decode("utf-8")

image_b64 = encode_image("image.jpg")

llm = ChatOllama(
    model="bakllava",
    temperature=0
)

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Describe this image in detail."
        },
        {
            "type": "image_url",
            "image_url": f"data:image/jpeg;base64,{image_b64}"
        }
    ]
)

response = llm.invoke([message])

print(response.content)

OpenAI
Claude
Gemini google
Groq
Openrouter
Huggingface

input   output
Text -> Text
Text -> Image
Image -> Text
Image -> Image(not required)
Text-> Audio
Audio -> Text
Audio -> Audio(not required)
text -> video
video -> text

you have to prepare a new notebook
where use a different model(which i have not use in the live class) related to different modality
and then generate a output
once you are successed in the notebook

create a webapp which is taking a input in every modality(text, image, audio, video)
and generate a output accordingly
